<a href="https://colab.research.google.com/github/LizaHam123/sentiment-analysis-algerie-telecom/blob/main/Extraction_commentaire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installation complète pour Google Colab
!pip install selenium

# Installation de Chrome et ChromeDriver
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver

# Créer un lien symbolique
!ln -sf /usr/bin/chromium-browser /usr/bin/google-chrome
!ln -sf /usr/bin/chromedriver /usr/local/bin/chromedriver

print("✅ Installation terminée")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException, TimeoutException
import pandas as pd
import time
import random
import os
from datetime import datetime, timedelta
import re

def create_driver():
    chrome_options = Options()

    # Options essentielles pour Google Colab
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-plugins")
    chrome_options.add_argument("--disable-images")
    chrome_options.add_argument("--disable-javascript")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36")

    # Désactiver les notifications
    prefs = {"profile.default_content_setting_values.notifications": 2}
    chrome_options.add_experimental_option("prefs", prefs)

    try:
        # Essayer avec le service
        service = Service('/usr/bin/chromedriver')
        driver = webdriver.Chrome(service=service, options=chrome_options)
        print(" Driver créé avec service")
        return driver
    except Exception as e:
        print(f" Erreur avec service: {e}")
        try:
            # Essayer sans service
            driver = webdriver.Chrome(options=chrome_options)
            print(" Driver créé sans service")
            return driver
        except Exception as e2:
            print(f" Erreur complète: {e2}")
            return None

# Test du driver
driver = create_driver()
if driver:
    print(" Test du driver réussi")
    driver.quit()
else:
    print(" Impossible de créer le driver")

 Erreur avec service: Message: Service /usr/bin/chromedriver unexpectedly exited. Status code was: 1

 Driver créé sans service
 Test du driver réussi


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException, TimeoutException
import pandas as pd
import time
import random
import os
from datetime import datetime, timedelta
import re

# Configuration du navigateur pour Google Colab
def create_driver():
    """Crée un driver Chrome robuste pour Google Colab"""
    chrome_options = Options()

    # Options pour Google Colab
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-notifications")

    prefs = {"profile.default_content_setting_values.notifications": 2}
    chrome_options.add_experimental_option("prefs", prefs)

    # Créer sans service (comme ça marche pour vous)
    driver = webdriver.Chrome(options=chrome_options)
    return driver

# Initialisation du driver
driver = None

# Données de connexion
FB_EMAIL = "arbidkawther@gmail.com"
FB_PASSWORD = "mDk65enY&"

def random_delay(min_sec=1, max_sec=3):
    time.sleep(random.uniform(min_sec, max_sec))

def parse_facebook_date(date_text):
    """Parse les dates Facebook et retourne un objet datetime"""
    try:
        # Nettoyer le texte de date
        date_text = date_text.strip().lower()

        # Gérer les dates relatives
        if "maintenant" in date_text or "à l'instant" in date_text:
            return datetime.now()
        elif "min" in date_text:
            minutes = int(re.search(r'(\d+)', date_text).group(1))
            return datetime.now() - timedelta(minutes=minutes)
        elif "h" in date_text and "hier" not in date_text:
            hours = int(re.search(r'(\d+)', date_text).group(1))
            return datetime.now() - timedelta(hours=hours)
        elif "hier" in date_text:
            return datetime.now() - timedelta(days=1)

        # Gérer les dates
        if "/" in date_text:
            parts = date_text.split("/")
            day = int(parts[0])
            month = int(parts[1])
            year = int(parts[2]) if len(parts) > 2 else datetime.now().year
            return datetime(year, month, day)

        # Mois en français
        mois_fr = {
            "janvier": 1, "février": 2, "mars": 3, "avril": 4, "mai": 5, "juin": 6,
            "juillet": 7, "août": 8, "septembre": 9, "octobre": 10, "novembre": 11, "décembre": 12
        }

        for mois_nom, mois_num in mois_fr.items():
            if mois_nom in date_text:
                day = int(re.search(r'(\d+)', date_text).group(1))
                year = datetime.now().year
                # Chercher l'année dans le texte
                year_match = re.search(r'20\d{2}', date_text)
                if year_match:
                    year = int(year_match.group())
                return datetime(year, mois_num, day)

        return None

    except Exception as e:
        print(f"Erreur parsing date '{date_text}': {e}")
        return None

def load_existing_comments(filename='commentaires_multiples_posts.csv'):
    """Charge les commentaires existants depuis le fichier CSV"""
    if os.path.exists(filename):
        df = pd.read_csv(filename)
        return set(df['Comment'].tolist()) if 'Comment' in df.columns else set()
    return set()

def save_comments(new_comments, filename='commentaires_multiples_posts.csv'):
    """Sauvegarde les commentaires en ajoutant aux existants (seulement le texte)"""
    if not new_comments:
        return 0

    # Convertir en format simple avec seulement les commentaires
    comments_only = [{'Comment': comment['Comment']} for comment in new_comments]

    if os.path.exists(filename):
        existing_df = pd.read_csv(filename)
        updated_df = pd.concat([existing_df, pd.DataFrame(comments_only)], ignore_index=True)
    else:
        updated_df = pd.DataFrame(comments_only)

    # Supprimer les doublons et sauvegarder
    updated_df = updated_df.drop_duplicates(subset=['Comment'])
    updated_df.to_csv(filename, index=False, encoding='utf-8-sig')
    return len(updated_df)

def facebook_login():
    try:
        driver.get("https://www.facebook.com/")
        random_delay(2, 4)

        # Accepter les cookies si nécessaire
        try:
            cookie_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-cookiebanner="accept_button"]')))
            cookie_btn.click()
            random_delay()
        except:
            pass

        # Saisie des identifiants
        email_field = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "input[name='email']")))
        email_field.clear()
        email_field.send_keys(FB_EMAIL)

        password_field = driver.find_element(By.CSS_SELECTOR, "input[name='pass']")
        password_field.clear()
        password_field.send_keys(FB_PASSWORD)

        # Connexion
        login_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
        login_button.click()

        WebDriverWait(driver, 15).until(
            lambda d: "facebook.com" in d.current_url and "login" not in d.current_url)
        print(" Connexion réussie")
        return True

    except Exception as e:
        print(f" Échec de la connexion: {e}")
        return False

def scroll_page():
    """Fait défiler la page pour charger plus de posts"""
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    random_delay(2, 4)

def get_posts_in_date_range(start_date, end_date, page_url="https://www.facebook.com/AlgerieTelecom"):
    """Récupère tous les posts dans la période spécifiée"""
    print(f" Recherche des posts entre {start_date.strftime('%d/%m/%Y')} et {end_date.strftime('%d/%m/%Y')}")

    driver.get(page_url)
    time.sleep(5)

    posts_data = []
    scroll_count = 0
    max_scrolls = 50  # Limite pour éviter les boucles infinies

    while scroll_count < max_scrolls:
        # Chercher tous les posts visibles
        try:
            post_containers = driver.find_elements(By.CSS_SELECTOR, "div[data-pagelet*='FeedUnit']")

            for container in post_containers:
                try:
                    # Chercher la date du post
                    date_elements = container.find_elements(By.CSS_SELECTOR, "a[role='link'] span")
                    post_date = None

                    for elem in date_elements:
                        date_text = elem.get_attribute('aria-label') or elem.text
                        if date_text:
                            post_date = parse_facebook_date(date_text)
                            if post_date:
                                break

                    if not post_date:
                        continue

                    # Vérifier si la date est dans la plage
                    if start_date <= post_date <= end_date:
                        # Récupérer le lien du post
                        post_links = container.find_elements(By.CSS_SELECTOR, "a[href*='/posts/']")
                        if post_links:
                            post_url = post_links[0].get_attribute('href')

                            # Vérifier si on a déjà ce post
                            if not any(p['url'] == post_url for p in posts_data):
                                # Récupérer le texte du post
                                post_text = ""
                                try:
                                    text_elements = container.find_elements(By.CSS_SELECTOR, "div[data-ad-preview='message']")
                                    if text_elements:
                                        post_text = text_elements[0].text[:100] + "..."
                                except:
                                    pass

                                posts_data.append({
                                    'url': post_url,
                                    'date': post_date,
                                    'preview': post_text
                                })
                                print(f" Post trouvé: {post_date.strftime('%d/%m/%Y')} - {post_text}")

                    elif post_date < start_date:
                        # Si on trouve un post plus ancien que start_date, on peut arrêter
                        print(f"🔚 Post trop ancien trouvé ({post_date.strftime('%d/%m/%Y')}), arrêt de la recherche")
                        return posts_data

                except Exception as e:
                    continue

            # Faire défiler pour charger plus de posts
            scroll_page()
            scroll_count += 1

            if scroll_count % 10 == 0:
                print(f" Défilement {scroll_count}/{max_scrolls} - {len(posts_data)} posts trouvés")

        except Exception as e:
            print(f" Erreur lors de la recherche: {e}")
            break

    print(f" Recherche terminée: {len(posts_data)} posts trouvés")
    return posts_data

def extract_comments_from_post(post_url, max_comments_per_post=100):
    """Extrait les commentaires d'un post spécifique"""
    print(f"💬 Extraction des commentaires de: {post_url}")

    driver.get(post_url)
    time.sleep(3)

    comments_data = []

    try:
        # Faire défiler et charger les commentaires
        scroll_attempts = 0
        max_scroll_attempts = 20

        while len(comments_data) < max_comments_per_post and scroll_attempts < max_scroll_attempts:
            # Chercher le bouton "Voir plus de commentaires"
            try:
                more_comments = driver.find_element(By.XPATH,
                    "//span[contains(text(), 'Voir plus de commentaires') or contains(text(), 'Afficher les commentaires précédents')]")
                driver.execute_script("arguments[0].click();", more_comments)
                time.sleep(2)
            except:
                pass

            # Faire défiler
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            # Extraire les commentaires visibles
            comment_containers = driver.find_elements(By.CSS_SELECTOR, "div[role='article']")

            for container in comment_containers:
                try:
                    # Récupérer le nom (pour vérifier que ce n'est pas Algérie Télécom)
                    name_element = container.find_element(By.CSS_SELECTOR, "span.xt0psk2")
                    name = name_element.text.strip()

                    # Ignorer les commentaires d'Algérie Télécom
                    if "Algérie Télécom" in name or "إتصالات الجزائر" in name:
                        continue

                    # Récupérer le commentaire
                    comment_element = container.find_element(By.CSS_SELECTOR, "div.x1lliihq.xjkvuk6.x1iorvi4")
                    comment_text = comment_element.text.strip()

                    # Éviter les doublons
                    if comment_text and not any(c['Comment'] == comment_text for c in comments_data):
                        comments_data.append({
                            'Name': name,
                            'Comment': comment_text,
                            'Post_URL': post_url
                        })

                        if len(comments_data) >= max_comments_per_post:
                            break

                except (NoSuchElementException, StaleElementReferenceException):
                    continue

            scroll_attempts += 1

            if len(comments_data) % 10 == 0 and len(comments_data) > 0:
                print(f"    {len(comments_data)} commentaires extraits...")

    except Exception as e:
        print(f" Erreur lors de l'extraction des commentaires: {e}")

    print(f" {len(comments_data)} commentaires extraits de ce post")
    return comments_data

def extract_comments_from_multiple_posts(start_date_str, end_date_str, max_comments_per_post=100):
    """Fonction principale pour extraire les commentaires de plusieurs posts"""
    global driver

    try:
        # Créer le driver
        driver = create_driver()
        print(" Driver créé avec succès")

        # Convertir les dates
        start_date = datetime.strptime(start_date_str, "%d/%m/%Y")
        end_date = datetime.strptime(end_date_str, "%d/%m/%Y")

        print(f" Début de l'extraction pour la période: {start_date_str} à {end_date_str}")

        # Se connecter
        if not facebook_login():
            return

        # Trouver tous les posts dans la période
        posts = get_posts_in_date_range(start_date, end_date)

        if not posts:
            print(" Aucun post trouvé dans cette période")
            return

        # Extraire les commentaires de chaque post
        all_comments = []
        existing_comments = load_existing_comments()

        for i, post in enumerate(posts, 1):
            print(f"\n Traitement du post {i}/{len(posts)}")

            post_comments = extract_comments_from_post(post['url'], max_comments_per_post)

            # Filtrer les nouveaux commentaires et garder seulement le texte
            new_comments = []
            for comment in post_comments:
                if comment['Comment'] not in existing_comments:
                    new_comments.append({'Comment': comment['Comment']})
                    existing_comments.add(comment['Comment'])

            all_comments.extend(new_comments)
            print(f"    {len(new_comments)} nouveaux commentaires ajoutés")

            # Pause entre les posts
            random_delay(3, 6)

        # Sauvegarder tous les commentaires
        if all_comments:
            total_count = save_comments(all_comments)
            print(f"\n EXTRACTION TERMINÉE!")
            print(f" {len(all_comments)} nouveaux commentaires extraits")
            print(f" Total dans le fichier: {total_count} commentaires uniques")
        else:
            print("\n Aucun nouveau commentaire trouvé")

    except Exception as e:
        print(f" Erreur principale: {e}")
    finally:
        if driver:
            driver.quit()
            print("🔚 Driver fermé")

if __name__ == "__main__":
    # Configuration de l'extraction
    START_DATE = "01/04/2024"
    END_DATE = "03/04/2024"
    MAX_COMMENTS_PER_POST = 20

    try:
        extract_comments_from_multiple_posts(START_DATE, END_DATE, MAX_COMMENTS_PER_POST)

    except Exception as e:
        print(f" Erreur critique: {e}")
    finally:
        if driver:
            driver.quit()
        print(" Navigateur fermé")

 Driver créé avec succès
 Début de l'extraction pour la période: 01/04/2024 à 03/04/2024
 Connexion réussie
 Recherche des posts entre 01/04/2024 et 03/04/2024
 Défilement 10/50 - 0 posts trouvés
 Défilement 20/50 - 0 posts trouvés
 Défilement 30/50 - 0 posts trouvés
 Défilement 40/50 - 0 posts trouvés
 Défilement 50/50 - 0 posts trouvés
 Recherche terminée: 0 posts trouvés
 Aucun post trouvé dans cette période


🔚 Driver fermé
 Navigateur fermé
